In [ ]:
# 設參數、環境

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import random

DATA_DIR = r"C:\Users\User\Desktop\高三專題\數據\ML\data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的裝置: {DEVICE}")

# ==== 實驗參數設定 ====
SEEDS = [42, 123, 456, 789, 999]
BATCH_SIZE = 128
EPOCHS_DEFAULT = 400
LR_DEFAULT = 1e-4

groups = {
    "Bend": ["ba", "bc"],
    "RightAngle": ["ra", "rc"],
    "Straight": ["s1", "s2"],
    "Mixed": ["ba", "bc", "ra", "rc", "s1", "s2"]
}

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)



In [ ]:
# 資料載入

def load_and_split_group(group_prefixes, seed=42):
    train_X_list, test_X_list = [], []
    train_y_list, test_y_list = [], []
    
    for prefix in group_prefixes:
        X_path = os.path.join(DATA_DIR, f"{prefix}.X.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y.npy")
        
        if os.path.exists(X_path) and os.path.exists(y_path):
            X = np.load(X_path)
            y = np.load(y_path)
            
            # 🔥 關鍵：針對單一檔案先切開
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=seed, shuffle=True
            )
            
            train_X_list.append(X_train)
            test_X_list.append(X_test)
            train_y_list.append(y_train)
            test_y_list.append(y_test)
            
    if not train_X_list: return None, None, None, None
    
    # 🔥 最後才把它們組合起來
    X_train_all = np.concatenate(train_X_list, axis=0).astype(np.float32)
    X_test_all = np.concatenate(test_X_list, axis=0).astype(np.float32)
    y_train_all = np.concatenate(train_y_list, axis=0).astype(np.float32)
    y_test_all = np.concatenate(test_y_list, axis=0).astype(np.float32)
    
    return X_train_all, X_test_all, y_train_all, y_test_all

def load_and_split_group_linear(group_prefixes, seed=42):
    train_X_list, test_X_list = [], []
    train_y_list, test_y_list = [], []
    
    for prefix in group_prefixes:
        X_path = os.path.join(DATA_DIR, f"{prefix}.X_linear.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y_linear.npy")
        
        if os.path.exists(X_path) and os.path.exists(y_path):
            X = np.load(X_path)
            y = np.load(y_path)
            
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=seed, shuffle=True
            )
            
            train_X_list.append(X_train)
            test_X_list.append(X_test)
            train_y_list.append(y_train)
            test_y_list.append(y_test)
            
    if not train_X_list: return None, None, None, None
    
    X_train_all = np.concatenate(train_X_list, axis=0).astype(np.float32)
    X_test_all = np.concatenate(test_X_list, axis=0).astype(np.float32)
    y_train_all = np.concatenate(train_y_list, axis=0).astype(np.float32)
    y_test_all = np.concatenate(test_y_list, axis=0).astype(np.float32)
    
    return X_train_all, X_test_all, y_train_all, y_test_all

def load_and_cheat_split_group(group_prefixes):
    # 作弊測試也必須同步更新，確保拿出來考的 Test 題目完全一樣
    X_all_list, y_all_list = [], []
    test_X_list, test_y_list = [], []
    
    for prefix in group_prefixes:
        X_path = os.path.join(DATA_DIR, f"{prefix}.X.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y.npy")
        
        if os.path.exists(X_path) and os.path.exists(y_path):
            X = np.load(X_path)
            y = np.load(y_path)
            
            # 作弊：Train 包含所有資料
            X_all_list.append(X)
            y_all_list.append(y)
            
            # 使用新邏輯抽出 Test 題目
            _, X_test, _, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, shuffle=True
            )
            test_X_list.append(X_test)
            test_y_list.append(y_test)
            
    if not X_all_list: return None, None, None, None
    
    X_train_cheat = np.concatenate(X_all_list, axis=0).astype(np.float32)
    y_train_cheat = np.concatenate(y_all_list, axis=0).astype(np.float32)
    
    X_test_cheat = np.concatenate(test_X_list, axis=0).astype(np.float32)
    y_test_cheat = np.concatenate(test_y_list, axis=0).astype(np.float32)
    
    return X_train_cheat, X_test_cheat, y_train_cheat, y_test_cheat


In [ ]:
# 模型定義

class LinearModel(nn.Module):
    def __init__(self, input_dim=50*24, output_dim=2):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc(x)

class InstantLinearModel(nn.Module):
    def __init__(self, input_dim=24, output_dim=2):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.fc(x)

class NNModel(nn.Module):
    def __init__(self, input_dim=50*24, output_dim=2):
        super().__init__()
        # 雙層隱藏層漏斗結構
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

class CNNModel(nn.Module):
    def __init__(self, num_features=24, output_dim=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 12, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class LSTMModel(nn.Module):
    def __init__(self, input_dim=24, hidden_dim=64, num_layers=1, output_dim=2):
        super().__init__()
        # 拔掉所有煞車，使用最原汁原味的 LSTM
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)
        


In [ ]:
# 訓練函數定義

def train_quiet(model, X_train, y_train, epochs, lr, batch_size):
    model = model.to(DEVICE)
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    criterion = nn.L1Loss(reduction='mean')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model

def train_cheat_custom_lr(model, X_train, y_train, epochs, batch_size=32):
    model = model.to(DEVICE)
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    train_criterion = nn.L1Loss(reduction='mean')
    eval_criterion = nn.L1Loss(reduction='sum')
    
    # 初始學習率 1e-3
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    lr_reduced = False
    
    model.train()
    for epoch in range(epochs):
        total_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            loss = train_criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            # 計算 Sum Loss 來判斷條件
            total_train_loss += eval_criterion(outputs, batch_y).item()
            
        # 🔥 條件觸發：如果 Loss 降到 200 以下，且還沒降速過
        if total_train_loss < 270 and not lr_reduced:
            for param_group in optimizer.param_groups:
                param_group['lr'] = 1e-4
            lr_reduced = True
            print(f"    [Epoch {epoch+1:03d}] ⚡觸發降速！Train Loss = {total_train_loss:.2f} < 200，學習率已切換為 1e-4")
            
        # 為了怕執行太久以為當機，每 50 個 Epoch 稍微報備一下進度
        if (epoch + 1) % 50 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"    [Epoch {epoch+1:03d}/{epochs}] Train Loss: {total_train_loss:.2f} | 學習率: {current_lr}")
            
    return model


def evaluate_loss(model, X_test, y_test, batch_size=32):
    model.eval()
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    criterion = nn.L1Loss(reduction='sum')
    total_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            total_loss += criterion(outputs, batch_y).item()
    return total_loss

def calculate_improvement(test_loss, y_test):
    # 基準：全部猜 0 的 L1 總誤差
    baseline_loss = np.sum(np.abs(y_test))
    return (1 - (test_loss / baseline_loss)) * 100




In [ ]:
# Part 1: 四個模型 x 三種獨立步態

print("========== Part 1: 四個模型 x 三種獨立步態 (5 Seed 平均進步百分比) ==========\n")

target_gaits = ["Bend", "RightAngle", "Straight"]
models_config = {
    "Linear": (LinearModel, EPOCHS_DEFAULT, LR_DEFAULT),
    "NN": (NNModel, EPOCHS_DEFAULT, LR_DEFAULT),
    "CNN": (CNNModel, EPOCHS_DEFAULT, LR_DEFAULT),
    "LSTM": (LSTMModel, EPOCHS_DEFAULT, LR_DEFAULT) # LSTM 學習率在函數內控制
}

results_part1 = []

for gait in target_gaits:
    print(f"\n--- 正在測試步態: {gait} ---")
    for model_name, (ModelClass, epochs, lr) in models_config.items():
        imp_list = []
        for seed in SEEDS:
            set_seed(seed)
            X_train, X_test, y_train, y_test = load_and_split_group(groups[gait], seed)
            
            model = ModelClass()
            trained_model = train_quiet(model, X_train, y_train, epochs, lr, BATCH_SIZE)
                
            test_loss = evaluate_loss(trained_model, X_test, y_test, BATCH_SIZE)
            imp = calculate_improvement(test_loss, y_test)
            imp_list.append(imp)
            
        mean_imp = np.mean(imp_list)
        std_imp = np.std(imp_list)
        results_part1.append({"Gait": gait, "Model": model_name, "Mean_Imp(%)": mean_imp, "Std_Imp(%)": std_imp})
        print(f"[{model_name.ljust(6)}] 平均進步: {mean_imp:6.2f}% (± {std_imp:5.2f}%)")

df_part1 = pd.DataFrame(results_part1)



In [ ]:
# Part 2: 四個模型 x 混合步態

print("========== Part 2: 四個模型 x 混合步態 Mixed Gait (5 Seed 平均進步百分比) ==========\n")

results_part2 = []

for model_name, (ModelClass, epochs, lr) in models_config.items():
    imp_list = []
    for seed in SEEDS:
        set_seed(seed)
        X_train, X_test, y_train, y_test = load_and_split_group(groups["Mixed"], seed)
        
        model = ModelClass()
        trained_model = train_quiet(model, X_train, y_train, epochs, lr, BATCH_SIZE)
            
        test_loss = evaluate_loss(trained_model, X_test, y_test, BATCH_SIZE)
        imp = calculate_improvement(test_loss, y_test)
        imp_list.append(imp)
        
    mean_imp = np.mean(imp_list)
    std_imp = np.std(imp_list)
    results_part2.append({"Model": model_name, "Mean_Imp(%)": mean_imp, "Std_Imp(%)": std_imp})
    print(f"[{model_name.ljust(6)}] 混合步態 平均進步: {mean_imp:6.2f}% (± {std_imp:5.2f}%)")

df_part2 = pd.DataFrame(results_part2)



In [ ]:
# Part 3: LSTM 跨步態預測矩陣

print("========== Part 3: LSTM 跨步態預測矩陣 (Cross-Gait Matrix) ==========\n")

target_domains = ["Bend", "RightAngle", "Straight", "Mixed"]
cross_matrix = {t: {} for t in target_domains}

for train_domain in target_domains:
    # 針對這個 train_domain，我們先把 5 個 seed 的模型訓練好並存起來
    print(f"\n--- 正在訓練專家: [{train_domain}] ---")
    trained_models_by_seed = {}
    
    for seed in SEEDS:
        set_seed(seed)
        X_train, _, y_train, _ = load_and_split_group(groups[train_domain], seed)
        model = LSTMModel()
        # 訓練並儲存
        trained_model = train_quiet(model, X_train, y_train, EPOCHS_DEFAULT, LR_DEFAULT, BATCH_SIZE)
        trained_models_by_seed[seed] = trained_model
        print(f"  > Seed {seed} 訓練完成")

    # 訓練好之後，直接拿這 5 個模型去考所有的考卷 (test_domain)
    for test_domain in target_domains:
        imp_list = []
        for seed in SEEDS:
            set_seed(seed)
            _, X_test, _, y_test = load_and_split_group(groups[test_domain], seed)
            
            # 直接拿出剛剛訓練好的模型來考試
            model_to_test = trained_models_by_seed[seed]
            test_loss = evaluate_loss(model_to_test, X_test, y_test, BATCH_SIZE)
            imp = calculate_improvement(test_loss, y_test)
            imp_list.append(imp)
            
        mean_imp = np.mean(imp_list)
        std_imp = np.std(imp_list)
        cross_matrix[train_domain][test_domain] = f"{mean_imp:.2f}% ±{std_imp:.2f}%"
        print(f"專家 [{train_domain}] 考 [{test_domain}] => 平均進步: {mean_imp:.2f}% (± {std_imp:.2f}%)")

df_cross = pd.DataFrame(cross_matrix).T
df_cross.index.name = '訓練資料 (Train)'
df_cross.columns.name = '測試資料 (Test)'
print("\n🏆 LSTM 跨步態預測交叉矩陣表 (數值為平均進步百分比 ± 標準差):")
display(df_cross)

In [ ]:
# Part 4: 極端物理極限測試

print("========== Part 4: 極端物理極限測試 ==========\n")

EPOCHS_CHEAT = 2000

# 1. LSTM Cheat Test (Train == All Data)
imp_cheat_list = []
for seed in SEEDS:
    set_seed(seed)
    # 使用 Cheat 函數，X_train 包含了所有的資料
    X_train, X_test, y_train, y_test = load_and_cheat_split_group(groups["Mixed"])
    
    model = LSTMModel()
    trained_model = train_cheat_custom_lr(model, X_train, y_train, EPOCHS_CHEAT, BATCH_SIZE)
    test_loss = evaluate_loss(trained_model, X_test, y_test, BATCH_SIZE)
    imp = calculate_improvement(test_loss, y_test)
    imp_cheat_list.append(imp)

mean_cheat = np.mean(imp_cheat_list)
std_cheat = np.std(imp_cheat_list)
print(f"👉 1. LSTM 作弊極限測試 (Mixed，看過考卷) => 平均進步: {mean_cheat:.2f}% (± {std_cheat:.2f}%)")

# 2. Instant Linear Test (Mixed)
imp_inst_list = []
for seed in SEEDS:
    set_seed(seed)
    X_train_lin, X_test_lin, y_train_lin, y_test_lin = load_and_split_group_linear(groups["Mixed"], seed)
    
    if X_train_lin is not None:
        model = InstantLinearModel()
        trained_model = train_quiet(model, X_train_lin, y_train_lin, EPOCHS_DEFAULT, LR_DEFAULT, BATCH_SIZE)
        test_loss = evaluate_loss(trained_model, X_test_lin, y_test_lin, BATCH_SIZE)
        imp = calculate_improvement(test_loss, y_test_lin)
        imp_inst_list.append(imp)

if imp_inst_list:
    mean_inst = np.mean(imp_inst_list)
    std_inst = np.std(imp_inst_list)
    print(f"👉 2. 瞬間線性模型極端測試 (Mixed，只有單點特徵) => 平均進步: {mean_inst:.2f}% (± {std_inst:.2f}%)")
else:
    print("找不到 _linear.npy 資料，無法執行瞬間線性測試。")

